# 03 — Silver: hava məlumatlarındakı NULL-ların doldurulması

Kod dəyişdirilməyib, yalnız hər blokun əvvəlinə **nə edir** və **niyə** izahı əlavə olunub.
**Ümumi məqsəd:** `silver.flights_clean` cədvəlində 2015-ci il üçün çatışmayan `WEATHER_*` dəyərlərini CSV-dən doldurmaq.

**Axın:** bağlantı → struktur → CSV oxu → uçuşları oxu → eşləşməni yoxla → staging-ə yüklə → `UPDATE` → nəticəni yoxla.

## 1. Setup: kitabxanalar və SQL Server bağlantısı

In [26]:
from common import *
engine = get_engine()

## 2. Import yolu və cədvəl strukturu sorğusu

In [12]:
import sys
sys.path.insert(0, "notebooks")
# Cədvəlin sütun adlarını və tiplərini al
query = text("""
    SELECT
        COLUMN_NAME,
        DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = :schema_name
      AND TABLE_NAME = :table_name
    ORDER BY ORDINAL_POSITION
""")

## 3. Sütun strukturunu oxu

In [13]:

columns_df = pd.read_sql(
    query,
    engine,
    params={
        "schema_name": "silver",
        "table_name": "flights_clean"
    }
)

## 4. Lazım olan sütunların siyahısı

In [ ]:
required_columns = [
    "FLIGHT_WEATHER_TIME",
    "YEAR",
    "MONTH",
    "DAY",
    "ORIGIN_AIRPORT",
    "WEATHER_TEMPERATURE",
    "WEATHER_CONDITION"
]

,COLUMN_NAME,DATA_TYPE
0,MONTH,varchar
1,DAY,smallint
4,ORIGIN_AIRPORT,varchar
30,FLIGHT_WEATHER_TIME,datetime
31,WEATHER_TEMPERATURE,real
34,WEATHER_CONDITION,real
39,YEAR,bigint


## 5. Hava CSV-ni oxu


In [16]:

# 1. Read the weather CSV
weather_path = "C:/Users/user/Desktop/Flight-delay-analysis-E2E/data/Flight/weather_missing_68_airports_2015.csv"
weather_df = pd.read_csv(weather_path)
weather_df["DATETIME"] = pd.to_datetime(weather_df["DATETIME"])

## 6. Uçuş sütunlarını seç

In [17]:

# 2. Load only the relevant flight columns
query = text("""
    SELECT
        FLIGHT_WEATHER_TIME,
        ORIGIN_AIRPORT,
        WEATHER_TEMPERATURE,
        WEATHER_PRECIPITATION,
        WEATHER_WIND_SPEED,
        WEATHER_CONDITION,
        WEATHER_SURFACE_PRESSURE,
        WEATHER_WIND_DIRECTION,
        WEATHER_SNOWFALL,
        WEATHER_WIND_GUSTS
    FROM silver.flights_clean
    WHERE YEAR = 2015
""")


## 7. Uçuş məlumatını yüklə və sətir saylarını müqayisə et

In [ ]:

flights_df = pd.read_sql(query, engine)
flights_df["FLIGHT_WEATHER_TIME"] = pd.to_datetime(
    flights_df["FLIGHT_WEATHER_TIME"]
)

# 3. Check whether weather keys are unique
weather_keys = weather_df[
    ["DATETIME", "ORIGIN_AIRPORT"]
]

print("Weather CSV rows:", len(weather_df))
print("Flight rows:", len(flights_df))


Weather CSV rows: 595680
Unique weather keys: True
Flight rows: 999657


## 8. Pandas-da qarşılı yoxlama (diaqnostika)

In [ ]:

# 4. Match by airport and exact date-time
weather_keys = weather_df.rename(
    columns={
        "DATETIME": "FLIGHT_WEATHER_TIME"
    }
)


In [ ]:

merged_df = flights_df.merge(
    weather_keys,
    on=["FLIGHT_WEATHER_TIME", "ORIGIN_AIRPORT"],
    how="left",
    suffixes=("_flight", "_csv"),
    indicator=True
)

print("\nMatch results:")
print(merged_df["_merge"].value_counts())


In [ ]:
weather_columns = [
    "WEATHER_TEMPERATURE",
    "WEATHER_PRECIPITATION",
    "WEATHER_WIND_SPEED",
    "WEATHER_CONDITION",
    "WEATHER_SURFACE_PRESSURE",
    "WEATHER_WIND_DIRECTION",
    "WEATHER_SNOWFALL",
    "WEATHER_WIND_GUSTS"
]

print("\nMissing values before filling:")
print(flights_df[weather_columns].isna().sum())

print("\nMatched rows with missing weather values:")
for col in weather_columns:
    csv_col = f"{col}_csv"
    if csv_col in merged_df.columns:
        count = (
            merged_df[col + "_flight"].isna()
            & merged_df[csv_col].notna()
        ).sum()
        print(f"{col}: {count}")

## 9. CSV-ni staging cədvəlinə yüklə və SQL tərəfində eşləşməni yoxla


In [ ]:
weather_path = "C:/Users/user/Desktop/Flight-delay-analysis-E2E/data/Flight/weather_missing_68_airports_2015.csv"
weather_df = pd.read_csv(weather_path)
weather_df["DATETIME"] = pd.to_datetime(weather_df["DATETIME"])

Weather CSV uploaded to staging table.
Rows uploaded:


,row_count
0,595680


,matched_flight_rows,matched_airports
0,226557,68


In [ ]:
weather_df.to_sql(
    name="weather_fill_staging_68",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False,
    chunksize=2000
)

print("Weather CSV uploaded to staging table.")


In [ ]:

# 3. Verify the uploaded row count
staging_count = pd.read_sql(
    text("SELECT COUNT(*) AS row_count FROM bronze.weather_fill_staging_68"),
    engine
)

print("Rows uploaded:")
display(staging_count)


In [ ]:
match_check = pd.read_sql(
    text("""
        SELECT
            COUNT_BIG(*) AS matched_flight_rows,
            COUNT_BIG(DISTINCT f.ORIGIN_AIRPORT) AS matched_airports
        FROM silver.flights_clean AS f
        INNER JOIN bronze.weather_fill_staging_68 AS w
            ON f.ORIGIN_AIRPORT = w.ORIGIN_AIRPORT
           AND f.FLIGHT_WEATHER_TIME = w.DATETIME
        WHERE f.YEAR = 2015
          AND f.WEATHER_TEMPERATURE IS NULL
    """),
    engine
)

display(match_check)

## 10. Doldurmadan əvvəl: CSV-də hər sütun nə qədər doludur

In [10]:

weather_columns = [
    "WEATHER_TEMPERATURE",
    "WEATHER_PRECIPITATION",
    "WEATHER_WIND_SPEED",
    "WEATHER_CONDITION",
    "WEATHER_SURFACE_PRESSURE",
    "WEATHER_WIND_DIRECTION",
    "WEATHER_SNOWFALL",
    "WEATHER_WIND_GUSTS",
]

query = text("""
    SELECT
        COUNT_BIG(*) AS matched_rows,
        SUM(CASE WHEN w.WEATHER_TEMPERATURE IS NOT NULL THEN 1 ELSE 0 END)
            AS temperature_available,
        SUM(CASE WHEN w.WEATHER_PRECIPITATION IS NOT NULL THEN 1 ELSE 0 END)
            AS precipitation_available,
        SUM(CASE WHEN w.WEATHER_WIND_SPEED IS NOT NULL THEN 1 ELSE 0 END)
            AS wind_speed_available,
        SUM(CASE WHEN w.WEATHER_CONDITION IS NOT NULL THEN 1 ELSE 0 END)
            AS condition_available,
        SUM(CASE WHEN w.WEATHER_SURFACE_PRESSURE IS NOT NULL THEN 1 ELSE 0 END)
            AS pressure_available,
        SUM(CASE WHEN w.WEATHER_WIND_DIRECTION IS NOT NULL THEN 1 ELSE 0 END)
            AS wind_direction_available,
        SUM(CASE WHEN w.WEATHER_SNOWFALL IS NOT NULL THEN 1 ELSE 0 END)
            AS snowfall_available,
        SUM(CASE WHEN w.WEATHER_WIND_GUSTS IS NOT NULL THEN 1 ELSE 0 END)
            AS wind_gusts_available
    FROM silver.flights_clean AS f
    INNER JOIN bronze.weather_fill_staging_68 AS w
        ON f.ORIGIN_AIRPORT = w.ORIGIN_AIRPORT
       AND f.FLIGHT_WEATHER_TIME = w.DATETIME
    WHERE f.YEAR = 2015
      AND (
          f.WEATHER_TEMPERATURE IS NULL
          OR f.WEATHER_PRECIPITATION IS NULL
          OR f.WEATHER_WIND_SPEED IS NULL
          OR f.WEATHER_CONDITION IS NULL
          OR f.WEATHER_SURFACE_PRESSURE IS NULL
          OR f.WEATHER_WIND_DIRECTION IS NULL
          OR f.WEATHER_SNOWFALL IS NULL
          OR f.WEATHER_WIND_GUSTS IS NULL
      )
""")

availability_df = pd.read_sql(query, engine)
display(availability_df)

,matched_rows,temperature_available,precipitation_available,wind_speed_available,condition_available,pressure_available,wind_direction_available,snowfall_available,wind_gusts_available
0,226557,226557,226557,226557,226557,226557,226557,226557,226557


## 11. UPDATE: NULL hava dəyərlərini doldur


In [19]:
from sqlalchemy import text

weather_columns = [
    "WEATHER_TEMPERATURE",
    "WEATHER_PRECIPITATION",
    "WEATHER_WIND_SPEED",
    "WEATHER_CONDITION",
    "WEATHER_SURFACE_PRESSURE",
    "WEATHER_WIND_DIRECTION",
    "WEATHER_SNOWFALL",
    "WEATHER_WIND_GUSTS",
]

# Update only NULL weather values for exact airport + datetime matches.
set_clause = ",\n".join(
    f"f.{col} = COALESCE(f.{col}, w.{col})"
    for col in weather_columns
)

update_query = text(f"""
    UPDATE f
    SET {set_clause}
    FROM silver.flights_clean AS f
    INNER JOIN bronze.weather_fill_staging_68 AS w
        ON f.ORIGIN_AIRPORT = w.ORIGIN_AIRPORT
       AND f.FLIGHT_WEATHER_TIME = w.DATETIME
    WHERE f.YEAR = 2015
      AND (
          f.WEATHER_TEMPERATURE IS NULL
          OR f.WEATHER_PRECIPITATION IS NULL
          OR f.WEATHER_WIND_SPEED IS NULL
          OR f.WEATHER_CONDITION IS NULL
          OR f.WEATHER_SURFACE_PRESSURE IS NULL
          OR f.WEATHER_WIND_DIRECTION IS NULL
          OR f.WEATHER_SNOWFALL IS NULL
          OR f.WEATHER_WIND_GUSTS IS NULL
      )
""")

with engine.begin() as connection:
    result = connection.execute(update_query)
    print("Updated rows:", result.rowcount)

Updated rows: 226557


## 12. Bütün silver cədvəlini pandas-a yüklə

In [ ]:
df = pd.read_sql(text("SELECT * FROM silver.flights_clean"),engine)

## 13. Qalan NULL-ları göstər


In [24]:
df.isna().sum()[df.isna().sum()>0]

DEPARTURE_DELAY         14867
ARRIVAL_DELAY           18142
AIR_SYSTEM_DELAY       817131
SECURITY_DELAY         817131
AIRLINE_DELAY          817131
LATE_AIRCRAFT_DELAY    817131
WEATHER_DELAY          817131
TAXI_OUT                15375
TAXI_IN                 15991
AIR_TIME                18142
IS_DELAYED              18142
dtype: int64

In [27]:

engine = get_engine()

query = """
SELECT
    s.name AS layer_name,
    t.name AS table_name
FROM sys.tables AS t
JOIN sys.schemas AS s
    ON t.schema_id = s.schema_id
WHERE s.name IN ('bronze', 'silver', 'gold')
ORDER BY s.name, t.name;
"""

with engine.connect() as connection:
    tables = pd.read_sql(text(query), connection)

if tables.empty:
    print("Heç bir cədvəl tapılmadı.")
else:
    print(tables.to_string(index=False))

layer_name              table_name
    bronze                airlines
    bronze                airports
    bronze                 flights
    bronze           flights_total
    bronze                 weather
    bronze weather_fill_staging_68
    silver           flights_clean
